In [ ]:
#r "BoSSSpad/BoSSSpad.dll"
using System;
using System.Collections.Generic;
using System.Linq;
using ilPSP;
using ilPSP.Utils;
using BoSSS.Platform;
using BoSSS.Foundation;
using BoSSS.Foundation.XDG;
using BoSSS.Foundation.Grid;
using BoSSS.Foundation.Grid.Classic;
using BoSSS.Foundation.IO;
using BoSSS.Solution;
using BoSSS.Solution.Control;
using BoSSS.Solution.GridImport;
using BoSSS.Solution.Statistic;
using BoSSS.Solution.Utils;
using BoSSS.Solution.AdvancedSolvers;
using BoSSS.Solution.Gnuplot;
using BoSSS.Application.BoSSSpad;
using BoSSS.Application.XNSE_Solver;
using static BoSSS.Application.BoSSSpad.BoSSSshell;
Init();

In [ ]:
using System.IO;
using BoSSS.Application.XNSE_Solver.PhysicalBasedTestcases;

In [ ]:
// var myBatch = ExecutionQueues[1];
// myBatch

In [ ]:
//BoSSSshell.WorkflowMgm.Init("RisingBubble2D", myBatch);

In [ ]:
var db = OpenOrCreateDatabase(@"P:\hpccluster\RisingBubble2D");

Opening existing database 'P:\hpccluster\RisingBubble2D'.


In [ ]:
// var sessions = BoSSSshell.WorkflowMgm.Sessions;
var sessions = db.Sessions;
sessions

#0: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_OneStepGaussAndStokes_restart3*	09/04/2024 12:53:16	d304f983...
#1: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart3*	09/02/2024 11:49:28	60901f69...
#2: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_ReInit100_testcase1	08/30/2024 09:22:51	44e79500...
#3: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_Picard_FastMarching_ReInit50_testcase1	08/30/2024 09:35:54	46f02668...
#4: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2*	08/29/2024 10:31:41	d3fbafe6...
#5: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withPreEnforcerActive_restart1*	08/28/2024 15:08:37	d55fdec7...
#6: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1*	08/27/2024 14:57:24	d9c4a861...


In [ ]:
List<ISessionInfo> evalSess = new List<ISessionInfo>();

In [ ]:
var sess = sessions.Pick(2);
sess

RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_ReInit100_testcase1	08/30/2024 09:22:51	44e79500...

In [ ]:
evalSess.Add(sess);

In [ ]:
//sess.Delete(true);

In [ ]:
//sess.Timesteps.TakeLast(10).Export().WithSupersampling(3).Do()

C:\Users\smuda\AppData\Local\BoSSS\plots\sessions\RisingBubble2D__RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2__d3fbafe6-263c-4031-ab07-a8453141fd8e

In [ ]:
// string studyName = "RB2D_fullDomain_20x40AMR0_k3_testcase1";
// foreach (var s in sessions) {
//     if (s.Name.Contains(studyName)) {
//         evalSess.Add(s);
//     }
// }
// evalSess

#0: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withUpdatedInterfaceCheck_restart2*	08/29/2024 10:31:41	d3fbafe6...
#1: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1_withPreEnforcerActive_restart1*	08/28/2024 15:08:37	d55fdec7...
#2: RisingBubble2D	RB2D_fullDomain_20x40AMR0_k3_testcase1*	08/27/2024 14:57:24	d9c4a861...


# Evaluation - rising bubble benchmark quantities

In [ ]:
public static void PerformPostProcessing(ISessionInfo evalS) {

    // set up log 
    var db = databases.Pick(0);
    TextWriter Log = db.Controller.DBDriver.FsDriver.GetNewLog(RisingBubble2DBenchmarkQuantities.LogfileName, evalS.ID);
    string header = String.Format("{0}\t{1}\t{2}\t{3}\t{4}\t{5}\t{6}\t{7}", "#timestep", "time", "area", "center of mass - x", "center of mass - y", "circularity", "mean velocity - x", "mean velocity - y");
    Log.WriteLine(header);
    Log.Flush();

    // perform postprocessing for each time step
    foreach(var tStep in evalS.Timesteps) {

        // set up LsTrk and velocity fields   
        SinglePhaseField phi = (SinglePhaseField)tStep.GetField("Phi");
        GridData grdData = (GridData)phi.GridDat; 
        LevelSet levelSet = new LevelSet(phi.Basis, "levelSet");
        levelSet.Acc(1.0, phi); 
        LevelSetTracker LsTrk = new LevelSetTracker(grdData, XQuadFactoryHelper.MomentFittingVariants.Saye,  1, new string[] {"A", "B"}, levelSet);
        LsTrk.UpdateTracker(tStep.PhysicalTime);

        int D = grdData.SpatialDimension;
        XDGField[] VelocityFields = new XDGField[D];
        VelocityFields[0] = (XDGField)tStep.GetField("VelocityX");
        VelocityFields[1] = (XDGField)tStep.GetField("VelocityY");

        // compute benchmark quantities
        var R = RisingBubbleBenchmarkQuantitites.ComputeBenchmarkQuantities_RisingBubble2D(LsTrk, VelocityFields);

        // write line to log
        string line = String.Format("{0}\t{1}\t{2}\t{3}\t{4}\t{5}\t{6}\t{7}", tStep.TimeStepNumber.MajorNumber, tStep.PhysicalTime, 
                        R.area, R.centerX, R.centerY, R.circularity, R.MeanVelocityX, R.MeanVelocityY);
        Log.WriteLine(line);
        Log.Flush();
    }

    // Dispose
    Log.Close();
    Log.Dispose();

}

In [ ]:
List<Plot2Ddata> evalData = evalSess.ReadLogDataForXNSE(RisingBubble2DBenchmarkQuantities.LogfileName);

Element at 0: time vs area
Element at 1: time vs center of mass - x
Element at 2: time vs center of mass - y
Element at 3: time vs circularity
Element at 4: time vs mean velocity - x
Element at 5: time vs mean velocity - y


In [ ]:
List<Plot2Ddata> benchmarkData = new List<Plot2Ddata>();
// 1: area 2: circularity 3: center y 4: rise velocity
benchmarkData.Add(evalData.ElementAt(0));
benchmarkData.Add(evalData.ElementAt(3));
benchmarkData.Add(evalData.ElementAt(2));
benchmarkData.Add(evalData.ElementAt(5));

In [ ]:
string logName = RisingBubble2DBenchmarkQuantities.LogfileName;
string evalName = null; 
string keyName = null;
var values = new string[] { "#timestep", "time", "area", "center of mass - x", "center of mass - y", "circularity", "mean velocity - x", "mean velocity - y" };

In [ ]:
List<Plot2Ddata> plotData = new List<Plot2Ddata>();

            int numberSessions = evalSess.Count();
            int numberValues = values.Count();      
            for (int vIdx = 2; vIdx < numberValues; vIdx++) {       

                double[][] times = new double[numberSessions][];
                double[][] valueDatas = new double[numberSessions][];

                // Read all data
                for (int j = 0; j < numberSessions; j++) {
                    string path = Path.Combine(evalSess.Pick(j).Database.Path, "sessions", evalSess.Pick(j).ID.ToString(), logName + ".txt");
                    string[] lines = File.ReadAllLines(path);

                    if (evalSess.Pick(j).RestartedFrom == Guid.Empty) { 
                   
                        double[] time = new double[lines.Length - 1];
                        double[] valueData = new double[lines.Length - 1];

                        for (int i = 0; i < lines.Length - 1; i++) {
                            time[i] = Convert.ToDouble(lines[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[1]);
                            valueData[i] = Convert.ToDouble(lines[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[vIdx]);
                        }
                        times[j] = time;
                        valueDatas[j] = valueData;

                    } else {

                        string pathR = @evalSess.Pick(j).Database.Path + "\\sessions\\" + evalSess.Pick(j).RestartedFrom + "\\" + logName + ".txt";
                        string[] linesR = File.ReadAllLines(pathR);

                        int len = (lines.Length - 1) + (linesR.Length - 1);
                        double[] time = new double[len];
                        double[] valueData = new double[len];
                        int iL = 0;
                        for (int i = 0; i < linesR.Length - 1; i++) {
                            time[iL] = Convert.ToDouble(linesR[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[1]);
                            valueData[iL] = Convert.ToDouble(linesR[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[vIdx]);
                            iL++;
                        }
                        for (int i = 0; i < lines.Length - 1; i++) {
                            time[iL] = Convert.ToDouble(lines[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[1]);
                            valueData[iL] = Convert.ToDouble(lines[i + 1].Split(new string[] { "\t" }, StringSplitOptions.RemoveEmptyEntries)[vIdx]);
                            iL++;
                        }

                        // remove doubled time steps 
                        List<double> rTime = new List<double>();
                        List<double> rValDat = new List<double>();
                        rTime.Add(time[len-1]);
                        rValDat.Add(valueData[len-1]);
                        for(int i = len-2; i >= 0; i--) {
                            if (time[i] < rTime.Last()) {
                                rTime.Add(time[i]);
                                rValDat.Add(valueData[i]);
                            }
                        }
                        rTime.Reverse();
                        rValDat.Reverse();

                        times[j] = rTime.ToArray();
                        valueDatas[j] = rValDat.ToArray();

                    }
                }

                // Build DataSet
                KeyValuePair<string, double[][]>[] dataRowsValue = new KeyValuePair<string, double[][]>[numberSessions];
                for (int i = 0; i < numberSessions; i++) {
                    string sessName;
                    if (evalName == null || keyName == null)
                        sessName = (evalSess.Pick(i).Name).Replace("_", "-");
                    else
                        sessName = evalName + (Convert.ToDouble(evalSess.Pick(i).KeysAndQueries[keyName])).ToString();

                    dataRowsValue[i] = new KeyValuePair<string, double[][]>(sessName, new double[][] { times[i], valueDatas[i] });
                }
                Console.WriteLine("Element at {0}: time vs {1}", vIdx - 2, values[vIdx]);
                plotData.Add(new Plot2Ddata(dataRowsValue));
            }

            evalData = plotData;

In [ ]:
//PerformPostProcessing(evalSess.Pick(0))

In [ ]:
// try {
//     evalData = evalSess.ReadLogDataForXNSE(RisingBubble2DBenchmarkQuantities.LogfileName);
// } catch {
//     Console.WriteLine("no log file found - do postprocessing on database session");
//     PerformPostProcessing(eval)
// }

no log file found


In [ ]:
public static Gnuplot GetBenchmarkQuantityOverTimeGnuplot(List<Plot2Ddata> data, int index, string name) {
    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "time";
    plt.Ylabel = name;
    
    var fmt = new PlotFormat();
    fmt.Style = Styles.Lines; 
    fmt.DashType = DashTypes.Solid;
    fmt.LineWidth = 2;
    fmt.LineColor = LineColors.Blue;

    plt.AddDataGroup(data.ElementAt(index).dataGroups[0], fmt);
    
    return plt.ToGnuplot();
}

In [ ]:
//ISessionInfoExtensions.PlotData(evalData.ElementAt(2), "time", "center of mass - y")
GetBenchmarkQuantityOverTimeGnuplot(evalData, 2, "center of mass - y").PlotSVG()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 1 
 
 
 
 
 1.1 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 center of mass - y 
 
 
 
 
 time 
 
 
 
 
 RB2D-fullDomain-20x40AMR0-k3-ReInit100-testcase1 
 
 
 
 
 RB2D-fullDomain-20x40AMR0-k3-ReInit100-testcase1 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M144.5,57.1 L197.9,57.1 M71.9,470.1 L72.4,470.1 L72.8,470.1 L73.3,470.1 L73.7,470.1 L74.2,470.1
 L74.6,470.0 L75.1,470.0 L75.5,470.0 L76.0,470.0 L76.4,470.0 L76.9,470.0 L77.3,470.0 L77.8,470.0
 L78.2,469.9 L78.7,469.9 L79.1,469.9 L79.6,469.9 L80.0,469.8 L80.5,469.8 L80.9,469.8 L81.4,469.8
 L81.8,469.7 L82.3,469.7 L82.8,469.7 L83.2,469.6 L83.7,469.6 L84.1,469.5 L84.6,469.5 L85.0,469.5
 L85.5,469.4 L85.9,469.4 L86.4,469.3 L86.8,469.3 L87.3,469.2 L87.7,469.2 L88.2,469.1 L88.6,469.1
 L89.1,469.0 L89.5,469.0 L90.0,468.9 L90.4,468.9 L90.9,468.8 L91.3,468.7 L91.8,468.7 L92.2,468.6
 L92.7,468.5 L93.2,468.5 L93.6,468.4 L94.1,468.3 L94.5,468.3 L95.0,468.2 L95.4,468.1 L95.9,468.1
 L96.3,468.0 L96.8,467.9 L97.2,467.8 L97.7,467.7 L98.1,467.7 L98.6,467.6 L99.0,467.5 L99.5,467.4
 L99.9,467.3 L100.4,467.2 L100.8,467.1 L101.3,467.1 L101.7,467.0 L102.2,466.9 L102.6,466.8 L103.1,466.7
 L103.5,466.6 L104.0,466.5 L104.5,466.4 L104.9,466.3 L105.4,466.2 L105.8,466.1 L106.3,466.0 L106.7,465.9
 L107.2,465.8 L107.6,465.6 L108.1,465.5 L108.5,465.4 L109.0,465.3 L109.4,465.2 L109.9,465.1 L110.3,465.0
 L110.8,464.9 L111.2,464.7 L111.7,464.6 L112.1,464.5 L112.6,464.4 L113.0,464.2 L113.5,464.1 L113.9,464.0
 L114.4,463.9 L114.9,463.7 L115.3,463.6 L115.8,463.5 L116.2,463.3 L116.7,463.2 L117.1,463.1 L117.6,463.1
 L118.0,462.9 L118.5,462.8 L118.9,462.7 L119.4,462.5 L119.8,462.4 L120.3,462.2 L120.7,462.1 L121.2,461.9
 L121.6,461.8 L122.1,461.6 L122.5,461.5 L123.0,461.3 L123.4,461.2 L123.9,461.0 L124.3,460.9 L124.8,460.7
 L125.3,460.6 L125.7,460.4 L126.2,460.3 L126.6,460.1 L127.1,459.9 L127.5,459.8 L128.0,459.6 L128.4,459.5
 L128.9,459.3 L129.3,459.1 L129.8,459.0 L130.2,458.8 L130.7,458.6 L131.1,458.4 L131.6,458.3 L132.0,458.1
 L132.5,457.9 L132.9,457.7 L133.4,457.6 L133.8,457.4 L134.3,457.2 L134.7,457.0 L135.2,456.9 L135.7,456.7
 L136.1,456.5 L136.6,456.3 L137.0,456.1 L137.5,455.9 L137.9,455.7 L138.4,455.6 L138.8,455.4 L139.3,455.2
 L139.7,455.0 L140.2,454.8 L140.6,454.6 L141.1,454.4 L141.5,454.2 L142.0,454.0 L142.4,453.8 L142.9,453.6
 L143.3,453.4 L143.8,453.2 L144.2,453.0 L144.7,452.8 L145.1,452.6 L145.6,452.4 L146.0,452.2 L146.5,452.0
 L147.0,451.8 L147.4,451.6 L147.9,451.3 L148.3,451.1 L148.8,450.9 L149.2,450.7 L149.7,450.5 L150.1,450.3
 L150.6,450.1 L151.0,449.8 L151.5,449.6 L151.9,449.4 L152.4,449.2 L152.8,449.0 L153.3,448.7 L153.7,448.5
 L154.2,448.3 L154.6,448.1 L155.1,447.8 L155.5,447.6 L156.0,447.4 L156.4,447.1 L156.9,446.9 L157.4,446.7
 L157.8,446.5 L158.3,446.2 L158.7,446.0 L159.2,445.7 L159.6,445.5 L160.1,445.3 L160.5,445.0 L161.0,444.8
 L161.4,444.6 L161.9,444.3 L162.3,444.1 L162.8,444.1 L163.2,443.8 L163.7,443.6 L164.1,443.3 L164.6,443.1
 L165.0,442.9 L165.5,442.6 L165.9,442.4 L166.4,442.1 L166.8,441.9 L167.3,441.6 L167.8,441.3 L168.2,441.1
 L168.7,440.8 L169.1,440.6 L169.6,440.3 L170.0,440.1 L170.5,439.8 L170.9,439.6 L171.4,439.3 L171.8,439.0
 L172.3,438.8 L172.7,438.5 L173.2,438.3 L173.6,438.0 L174.1,437.7 L174.5,437.5 L175.0,437.2 L175.4,436.9
 L175.9,436.7 L176.3,436.4 L176.8,436.1 L177.2,435.9 L177.7,435.6 L178.2,435.3 L178.6,435.0 L179.1,434.8
 L179.5,434.5 L180.0,434.2 L180.4,433.9 L180.9,433.7 L181.3,433.4 L181.8,433.1 L182.2,432

In [ ]:
//ISessionInfoExtensions.PlotData(evalData.ElementAt(3), "time", "circularity")
GetBenchmarkQuantityOverTimeGnuplot(evalData, 3, "circularity").PlotSVG()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.9 
 
 
 
 
 0.91 
 
 
 
 
 0.92 
 
 
 
 
 0.93 
 
 
 
 
 0.94 
 
 
 
 
 0.95 
 
 
 
 
 0.96 
 
 
 
 
 0.97 
 
 
 
 
 0.98 
 
 
 
 
 0.99 
 
 
 
 
 1 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 circularity 
 
 
 
 
 time 
 
 
 
 
 RB2D-fullDomain-20x40AMR0-k3-ReInit100-testcase1 
 
 
 
 
 RB2D-fullDomain-20x40AMR0-k3-ReInit100-testcase1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M144.5,57.1 L197.9,57.1 M80.2,36.1 L80.6,36.1 L81.1,36.1 L81.5,36.1 L82.0,36.1 L82.4,36.1
 L82.9,36.1 L83.3,36.1 L83.8,36.1 L84.2,36.1 L84.7,36.1 L85.1,36.1 L85.6,36.1 L86.0,36.1
 L86.5,36.1 L86.9,36.1 L87.3,36.1 L87.8,36.1 L88.2,36.1 L88.7,36.1 L89.1,36.1 L89.6,36.1
 L90.0,36.1 L90.5,36.1 L90.9,36.1 L91.4,36.1 L91.8,36.1 L92.3,36.1 L92.7,36.1 L93.2,36.1
 L93.6,36.1 L94.0,36.1 L94.5,36.1 L94.9,36.1 L95.4,36.1 L95.8,36.1 L96.3,36.1 L96.7,36.1
 L97.2,36.1 L97.6,36.1 L98.1,36.1 L98.5,36.1 L99.0,36.1 L99.4,36.1 L99.9,36.1 L100.3,36.1
 L100.7,36.1 L101.2,36.1 L101.6,36.1 L102.1,36.1 L102.5,36.1 L103.0,36.1 L103.4,36.1 L103.9,36.1
 L104.3,36.1 L104.8,36.1 L105.2,36.1 L105.7,36.1 L106.1,36.1 L106.5,36.1 L107.0,36.1 L107.4,36.1
 L107.9,36.1 L108.3,36.1 L108.8,36.1 L109.2,36.1 L109.7,36.1 L110.1,36.1 L110.6,36.1 L111.0,36.1
 L111.5,36.1 L111.9,36.1 L112.4,36.1 L112.8,36.1 L113.2,36.1 L113.7,36.1 L114.1,36.1 L114.6,36.1
 L115.0,36.1 L115.5,36.1 L115.9,36.1 L116.4,36.1 L116.8,36.1 L117.3,36.1 L117.7,36.1 L118.2,36.1
 L118.6,36.1 L119.1,36.1 L119.5,36.1 L119.9,36.1 L120.4,36.1 L120.8,36.1 L121.3,36.1 L121.7,36.1
 L122.2,36.1 L122.6,36.1 L123.1,36.1 L123.5,36.1 L124.0,36.1 L124.4,36.1 L124.9,36.1 L125.3,36.1
 L125.8,36.1 L126.2,36.1 L126.6,36.1 L127.1,36.1 L127.5,36.1 L128.0,36.1 L128.4,36.1 L128.9,36.1
 L129.3,36.1 L129.8,36.1 L130.2,36.1 L130.7,36.1 L131.1,36.1 L131.6,36.1 L132.0,36.1 L132.5,36.1
 L132.9,36.1 L133.3,36.1 L133.8,36.1 L134.2,36.1 L134.7,36.2 L135.1,36.2 L135.6,36.2 L136.0,36.2
 L136.5,36.2 L136.9,36.2 L137.4,36.2 L137.8,36.2 L138.3,36.2 L138.7,36.2 L139.2,36.2 L139.6,36.2
 L140.0,36.2 L140.5,36.2 L140.9,36.2 L141.4,36.2 L141.8,36.2 L142.3,36.2 L142.7,36.2 L143.2,36.2
 L143.6,36.2 L144.1,36.2 L144.5,36.2 L145.0,36.2 L145.4,36.2 L145.9,36.2 L146.3,36.2 L146.7,36.2
 L147.2,36.2 L147.6,36.2 L148.1,36.2 L148.5,36.2 L149.0,36.3 L149.4,36.3 L149.9,36.3 L150.3,36.3
 L150.8,36.3 L151.2,36.3 L151.7,36.3 L152.1,36.3 L152.5,36.3 L153.0,36.3 L153.4,36.3 L153.9,36.3
 L154.3,36.3 L154.8,36.3 L155.2,36.4 L155.7,36.4 L156.1,36.4 L156.6,36.4 L157.0,36.4 L157.5,36.4
 L157.9,36.4 L158.4,36.4 L158.8,36.4 L159.2,36.4 L159.7,36.5 L160.1,36.5 L160.6,36.5 L161.0,36.5
 L161.5,36.5 L161.9,36.5 L162.4,36.5 L162.8,36.6 L163.3,36.6 L163.7,36.6 L164.2,36.6 L164.6,36.6
 L165.1,36.6 L165.5,36.7 L165.9,36.7 L166.4,36.7 L166.8,36.7 L167.3,36.7 L167.7,36.8 L168.2,36.8
 L168.6,36.8 L169.1,36.8 L169.5,36.8 L170.0,36.8 L170.4,36.9 L170.9,36.9 L171.3,36.9 L171.8,36.9
 L172.2,37.0 L172.6,37.0 L173.1,37.0 L173.5,37.1 L174.0,37.1 L174.4,37.1 L174.9,37.1 L175.3,37.2
 L175.8,37.2 L176.2,37.3 L176.7,37.3 L177.1,37.3 L177.6,37.4 L178.0,37.4 L178.5,37.4 L178.9,37.5
 L179.3,37.5 L179.8,37.6 L180.2,37.6 L180.7,37.7 L181.1,37.7 L181.6,37.8 L182.0,37.8 L182.5,37.9
 L182.9,37.9 L183.4,38.0 L183.8,38.0 L184.3,38.1 L184.7,38.1 L185.2,38.2 L185.6,38.3 L186.0,38.3
 L186.5,38.4 L186.9,38.4 L187.4,38.5 L187.8,38.6 L188.3,38.6 L188.7,38.7 L189.2,38.8 L189.6,38.9
 L190.1,38.9 L190.5,39.0 L191.0,39.1 L191.4,39.2 L191.9,39.3 L192.3,39.4 L192.7,39.4 L193.2,39.5
 L193.6,39.6 L194.1,39.7 L194.5,39.8 L195.

In [ ]:
//ISessionInfoExtensions.PlotData(evalData.ElementAt(5), "time", "rise velocity")
GetBenchmarkQuantityOverTimeGnuplot(evalData, 5, "rise velocity").PlotSVG()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rise velocity 
 
 
 
 
 time 
 
 
 
 
 RB2D-fullDomain-20x40AMR0-k3-ReInit100-testcase1 
 
 
 
 
 RB2D-fullDomain-20x40AMR0-k3-ReInit100-testcase1 
 
 
 
	<path stroke='rgb(255, 0, 0)' d='M144.5,57.1 L197.9,57.1 M80.2,542.4 L80.6,540.2 L81.1,538.1 L81.5,535.9 L82.0,533.8 L82.4,531.6
 L82.9,529.5 L83.3,527.4 L83.8,525.3 L84.2,523.2 L84.7,521.1 L85.1,519.0 L85.6,516.9 L86.0,514.8
 L86.5,512.8 L86.9,510.7 L87.3,508.6 L87.8,506.6 L88.2,504.5 L88.7,502.5 L89.1,500.4 L89.6,498.4
 L90.0,496.4 L90.5,494.3 L90.9,492.3 L91.4,490.3 L91.8,488.3 L92.3,486.3 L92.7,484.3 L93.2,482.3
 L93.6,480.3 L94.0,478.3 L94.5,476.4 L94.9,474.4 L95.4,472.4 L95.8,470.4 L96.3,468.5 L96.7,466.5
 L97.2,464.6 L97.6,462.6 L98.1,460.7 L98.5,458.8 L99.0,456.8 L99.4,454.9 L99.9,453.0 L100.3,451.0
 L100.7,449.1 L101.2,447.2 L101.6,445.3 L102.1,443.4 L102.5,441.5 L103.0,439.6 L103.4,437.7 L103.9,435.8
 L104.3,434.0 L104.8,432.1 L105.2,430.2 L105.7,428.3 L106.1,426.5 L106.5,424.6 L107.0,422.8 L107.4,420.9
 L107.9,419.0 L108.3,417.2 L108.8,415.4 L109.2,413.5 L109.7,411.7 L110.1,409.9 L110.6,408.0 L111.0,406.2
 L111.5,404.4 L111.9,402.6 L112.4,400.8 L112.8,399.0 L113.2,397.2 L113.7,395.4 L114.1,393.6 L114.6,391.8
 L115.0,390.0 L115.5,388.2 L115.9,386.4 L116.4,384.7 L116.8,382.9 L117.3,381.1 L117.7,379.4 L118.2,377.6
 L118.6,375.8 L119.1,374.1 L119.5,372.3 L119.9,370.6 L120.4,368.9 L120.8,367.1 L121.3,365.4 L121.7,363.7
 L122.2,361.9 L122.6,360.2 L123.1,358.5 L123.5,356.8 L124.0,355.1 L124.4,353.4 L124.9,351.7 L125.3,350.0
 L125.8,348.3 L126.2,346.6 L126.6,344.9 L127.1,343.2 L127.5,341.5 L128.0,339.9 L128.4,338.2 L128.9,336.5
 L129.3,334.8 L129.8,333.2 L130.2,331.5 L130.7,329.9 L131.1,328.2 L131.6,326.6 L132.0,324.9 L132.5,323.3
 L132.9,321.7 L133.3,320.0 L133.8,318.4 L134.2,316.8 L134.7,315.2 L135.1,313.6 L135.6,312.0 L136.0,310.4
 L136.5,308.8 L136.9,307.2 L137.4,305.6 L137.8,304.0 L138.3,302.4 L138.7,300.8 L139.2,299.2 L139.6,297.7
 L140.0,296.1 L140.5,294.5 L140.9,293.0 L141.4,291.4 L141.8,289.8 L142.3,288.3 L142.7,286.7 L143.2,285.2
 L143.6,283.7 L144.1,282.1 L144.5,280.6 L145.0,279.1 L145.4,277.6 L145.9,276.0 L146.3,274.5 L146.7,273.0
 L147.2,271.5 L147.6,270.0 L148.1,268.5 L148.5,267.0 L149.0,265.6 L149.4,264.1 L149.9,262.6 L150.3,261.1
 L150.8,259.6 L151.2,258.2 L151.7,256.7 L152.1,255.3 L152.5,253.8 L153.0,252.4 L153.4,250.9 L153.9,249.5
 L154.3,248.1 L154.8,246.6 L155.2,245.2 L155.7,243.8 L156.1,242.4 L156.6,241.0 L157.0,239.6 L157.5,238.1
 L157.9,236.7 L158.4,235.4 L158.8,234.0 L159.2,232.6 L159.7,231.2 L160.1,229.8 L160.6,228.5 L161.0,227.1
 L161.5,225.7 L161.9,224.4 L162.4,223.0 L162.8,221.7 L163.3,220.4 L163.7,219.0 L164.2,217.7 L164.6,216.4
 L165.1,215.0 L165.5,213.7 L165.9,212.4 L166.4,211.1 L166.8,209.8 L167.3,208.5 L167.7,207.2 L168.2,205.9
 L168.6,204.7 L169.1,203.4 L169.5,202.1 L170.0,200.7 L170.4,199.5 L170.9,198.2 L171.3,196.9 L171.8,195.7
 L172.2,194.5 L172.6,193.2 L173.1,192.0 L173.5,190.8 L174.0,189.5 L174.4,188.3 L174.9,187.1 L175.3,185.9
 L175.8,184.7 L176.2,183.5 L176.7,182.3 L177.1,181.2 L177.6,180.0 L178.0,178.8 L178.5,177.6 L178.9,176.5
 L179.3,175.3 L179.8,174.2 L180.2,173.0 L180.7,171.9 L181.1,170.7 L181.6,169.6 L182.0,168.5 L182.5,167.4
 L182.9,166.3 L183.4,165.2 L183.8,164.1 L184.3,163.0 L184.7,161.9 L185.2,160.8 L185.6,159.7 L186.0,158.6
 L186.5,157.6 L186.9,156.5 L187.4,155.5 L187.8,154.4 L188.3,153.4 L188.7,152.3 L189.2,151.3 L189.6,150.3
 L190.1,149.2 L19

## Compare to Reference data

In [ ]:
string caseName = "testcase1";

In [ ]:
// g1 TU Dortmund (TP2D); g2 EPFL Lausanne (FreeLIFE); g3 Uni Magdebug (MooNMD)
string[] groups = new string[] {"TP2D", "FreeLIFE", "MooNMD"};
int[,] datLvl;
//int[] datLvl;
string datCase;
if(caseName.Equals("testcase1")) {
    datCase = "c1g";
    datLvl  = new int[,] {{4, 5, 6, 7}, {1, 2, 3, -1}, {1, 2, 3, 4}};    // testcase 1
    //datLvl  = new int[] {7, 3, 4};    // testcase 1
} 
if(caseName.Equals("testcase2")) {     
    datCase = "c2g";
    datLvl  = new int[,] {{4, 5, 6, 7, 8}, {1, 2, 3, -1, -1}, {2, 3, 4, -1, -1}};    // testcase 2
    //datLvl  = new int[] { 8, 3, 4};    // testcase 2
}
List<Plot2Ddata>[,] dataRef = new List<Plot2Ddata>[4,3];
for (int grp = 1; grp <= groups.Length; grp++) {
    List<Plot2Ddata>[] datSet = new List<Plot2Ddata>[4];
    // 1: area 2: circularity 3: center y 4: rise velocity
    for (int j = 0; j < 4; j++) {
        datSet[j] = new List<Plot2Ddata>();
    }

    int numL = datLvl.GetLength(1);
    for (int l = 0; l < numL; l++) {
        if(datLvl[grp-1,l] == -1)
            continue;
        // Read all data
        string dat  = datCase+grp+"l"+datLvl[grp - 1,l]+".txt";
        string path = @"D:\local\examplaes_XNSEdata\RisingBubble\referenceData_Featflow\data_bench_quantities\"+dat;
        string[] lines = File.ReadAllLines(path);
        double[] time = new double[lines.Length];
        double[][] valueDat = new double[4][];
        for(int j = 0; j < 4; j++)
            valueDat[j] = new double[lines.Length];

        for (int i = 0; i < lines.Length; i++) {
            //var datString = lines[i].Split(new string[] {" "}, StringSplitOptions.RemoveEmptyEntries);
            //Console.WriteLine("num split strings at 0: {0}", datString[0]);
            time[i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[0]);            
            for (int j = 0; j < 4; j++) {
                valueDat[j][i] = Convert.ToDouble(lines[i].Split(new string[] { " " }, StringSplitOptions.RemoveEmptyEntries)[j+1]);
            }
        }        
        // Build DataSet
        for (int j = 0; j < 4; j++) {
            string datName = groups[grp-1]+"-l"+datLvl[grp - 1,l];
            datSet[j].Add(new Plot2Ddata(new KeyValuePair<string, double[][]>(datName, new double[][] { time, valueDat[j] })));
        }
    }
    
    for (int j = 0; j < 4; j++) {
        dataRef[j,grp-1] = datSet[j];
    }
}

In [ ]:
// List<Plot2Ddata> evalDataRef = evalData;
// for (int i = 0; i < 3; i++) {
//     evalDataRef[0] = evalDataRef.ElementAt(0).Merge(dataRef[0,i].Last());
//     evalDataRef[3] = evalDataRef.ElementAt(3).Merge(dataRef[1,i].Last());
//     evalDataRef[2] = evalDataRef.ElementAt(2).Merge(dataRef[2,i].Last());
//     evalDataRef[5] = evalDataRef.ElementAt(5).Merge(dataRef[3,i].Last());
// }

In [ ]:
public static Gnuplot GetBenchmarkQuantityComparisonOverTimeGnuplot(List<Plot2Ddata> data, int index, string name, List<Plot2Ddata>[,] refData) {
    Plot2Ddata plt =  new Plot2Ddata();
    plt.Xlabel = "time";
    plt.Ylabel = name;
    
    var fmt = new PlotFormat();
    fmt.Style = Styles.Lines; 
    fmt.DashType = DashTypes.Solid;
    fmt.LineWidth = 2;
    fmt.LineColor = LineColors.Blue;

    var datgrp = data.ElementAt(index).dataGroups[0];
    plt.AddDataGroup("BoSSS", datgrp.Abscissas, datgrp.Values, fmt.CloneAs());

    fmt.LineColor = LineColors.Magenta;
    plt.AddDataGroup(refData[index, 0].Last().dataGroups[0], fmt.CloneAs());

    fmt.LineColor = LineColors.Green;
    plt.AddDataGroup(refData[index, 1].Last().dataGroups[0], fmt.CloneAs());

    fmt.LineColor = LineColors.Orange;
    plt.AddDataGroup(refData[index, 2].Last().dataGroups[0], fmt.CloneAs());
    
    return plt.ToGnuplot();
}

In [ ]:
//ISessionInfoExtensions.PlotData(evalDataRef.ElementAt(3), "time", "circularity")
GetBenchmarkQuantityComparisonOverTimeGnuplot(benchmarkData, 1, "circularity", dataRef).PlotSVG()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.9 
 
 
 
 
 0.91 
 
 
 
 
 0.92 
 
 
 
 
 0.93 
 
 
 
 
 0.94 
 
 
 
 
 0.95 
 
 
 
 
 0.96 
 
 
 
 
 0.97 
 
 
 
 
 0.98 
 
 
 
 
 0.99 
 
 
 
 
 1 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 circularity 
 
 
 
 
 time 
 
 
 
 
 BoSSS 
 
 
 
 
 BoSSS 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M555.2,57.1 L608.6,57.1 M80.2,36.1 L80.6,36.1 L81.0,36.1 L81.3,36.1 L81.7,36.1 L82.1,36.1
 L82.5,36.1 L82.9,36.1 L83.3,36.1 L83.6,36.1 L84.0,36.1 L84.4,36.1 L84.8,36.1 L85.2,36.1
 L85.6,36.1 L85.9,36.1 L86.3,36.1 L86.7,36.1 L87.1,36.1 L87.5,36.1 L87.9,36.1 L88.2,36.1
 L88.6,36.1 L89.0,36.1 L89.4,36.1 L89.8,36.1 L90.2,36.1 L90.5,36.1 L90.9,36.1 L91.3,36.1
 L91.7,36.1 L92.1,36.1 L92.4,36.1 L92.8,36.1 L93.2,36.1 L93.6,36.1 L94.0,36.1 L94.4,36.1
 L94.7,36.1 L95.1,36.1 L95.5,36.1 L95.9,36.1 L96.3,36.1 L96.7,36.1 L97.0,36.1 L97.4,36.1
 L97.8,36.1 L98.2,36.1 L98.6,36.1 L99.0,36.1 L99.3,36.1 L99.7,36.1 L100.1,36.1 L100.5,36.1
 L100.9,36.1 L101.3,36.1 L101.6,36.1 L102.0,36.1 L102.4,36.1 L102.8,36.1 L103.2,36.1 L103.6,36.1
 L103.9,36.1 L104.3,36.1 L104.7,36.1 L105.1,36.1 L105.5,36.1 L105.8,36.1 L106.2,36.1 L106.6,36.1
 L107.0,36.1 L107.4,36.1 L107.8,36.1 L108.1,36.1 L108.5,36.1 L108.9,36.1 L109.3,36.1 L109.7,36.1
 L110.1,36.1 L110.4,36.1 L110.8,36.1 L111.2,36.1 L111.6,36.1 L112.0,36.1 L112.4,36.1 L112.7,36.1
 L113.1,36.1 L113.5,36.1 L113.9,36.1 L114.3,36.1 L114.7,36.1 L115.0,36.1 L115.4,36.1 L115.8,36.1
 L116.2,36.1 L116.6,36.1 L116.9,36.1 L117.3,36.1 L117.7,36.1 L118.1,36.1 L118.5,36.1 L118.9,36.1
 L119.2,36.1 L119.6,36.1 L120.0,36.1 L120.4,36.1 L120.8,36.1 L121.2,36.1 L121.5,36.1 L121.9,36.1
 L122.3,36.1 L122.7,36.1 L123.1,36.1 L123.5,36.1 L123.8,36.1 L124.2,36.1 L124.6,36.1 L125.0,36.1
 L125.4,36.1 L125.8,36.1 L126.1,36.1 L126.5,36.1 L126.9,36.2 L127.3,36.2 L127.7,36.2 L128.1,36.2
 L128.4,36.2 L128.8,36.2 L129.2,36.2 L129.6,36.2 L130.0,36.2 L130.3,36.2 L130.7,36.2 L131.1,36.2
 L131.5,36.2 L131.9,36.2 L132.3,36.2 L132.6,36.2 L133.0,36.2 L133.4,36.2 L133.8,36.2 L134.2,36.2
 L134.6,36.2 L134.9,36.2 L135.3,36.2 L135.7,36.2 L136.1,36.2 L136.5,36.2 L136.9,36.2 L137.2,36.2
 L137.6,36.2 L138.0,36.2 L138.4,36.2 L138.8,36.2 L139.2,36.3 L139.5,36.3 L139.9,36.3 L140.3,36.3
 L140.7,36.3 L141.1,36.3 L141.4,36.3 L141.8,36.3 L142.2,36.3 L142.6,36.3 L143.0,36.3 L143.4,36.3
 L143.7,36.3 L144.1,36.3 L144.5,36.4 L144.9,36.4 L145.3,36.4 L145.7,36.4 L146.0,36.4 L146.4,36.4
 L146.8,36.4 L147.2,36.4 L147.6,36.4 L148.0,36.4 L148.3,36.5 L148.7,36.5 L149.1,36.5 L149.5,36.5
 L149.9,36.5 L150.3,36.5 L150.6,36.5 L151.0,36.6 L151.4,36.6 L151.8,36.6 L152.2,36.6 L152.5,36.6
 L152.9,36.6 L153.3,36.7 L153.7,36.7 L154.1,36.7 L154.5,36.7 L154.8,36.7 L155.2,36.8 L155.6,36.8
 L156.0,36.8 L156.4,36.8 L156.8,36.8 L157.1,36.8 L157.5,36.9 L157.9,36.9 L158.3,36.9 L158.7,36.9
 L159.1,37.0 L159.4,37.0 L159.8,37.0 L160.2,37.1 L160.6,37.1 L161.0,37.1 L161.4,37.1 L161.7,37.2
 L162.1,37.2 L162.5,37.3 L162.9,37.3 L163.3,37.3 L163.7,37.4 L164.0,37.4 L164.4,37.4 L164.8,37.5
 L165.2,37.5 L165.6,37.6 L165.9,37.6 L166.3,37.7 L166.7,37.7 L167.1,37.8 L167.5,37.8 L167.9,37.9
 L168.2,37.9 L168.6,38.0 L169.0,38.0 L169.4,38.1 L169.8,38.1 L170.2,38.2 L170.5,38.3 L170.9,38.3
 L171.3,38.4 L171.7,38.4 L172.1,38.5 L172.5,38.6 L172.8,38.6 L173.2,38.7 L173.6,38.8 L174.0,38.9
 L174.4,38.9 L174.8,39.0 L175.1,39.1 L175.5,39.2 L175.9,39.3 L176.3,39.4 L176.7,39.4 L177.0,39.5
 L177.4,39.6 L177.8,39.7 L178.2,39.8 L178.6,39.9 L179.0,40.0 L179.3,40.1 L179.7,40.2 L180.1,40.3
 L180.5,40.4 

In [ ]:
GetBenchmarkQuantityComparisonOverTimeGnuplot(benchmarkData, 2, "center of mass - y", dataRef).PlotSVG()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 0.4 
 
 
 
 
 0.5 
 
 
 
 
 0.6 
 
 
 
 
 0.7 
 
 
 
 
 0.8 
 
 
 
 
 0.9 
 
 
 
 
 1 
 
 
 
 
 1.1 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 center of mass - y 
 
 
 
 
 time 
 
 
 
 
 BoSSS 
 
 
 
 
 BoSSS 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M555.2,57.1 L608.6,57.1 M71.9,470.1 L72.3,470.1 L72.7,470.1 L73.1,470.1 L73.5,470.1 L73.8,470.1
 L74.2,470.0 L74.6,470.0 L75.0,470.0 L75.4,470.0 L75.8,470.0 L76.2,470.0 L76.6,470.0 L76.9,470.0
 L77.3,469.9 L77.7,469.9 L78.1,469.9 L78.5,469.9 L78.9,469.8 L79.3,469.8 L79.7,469.8 L80.0,469.8
 L80.4,469.7 L80.8,469.7 L81.2,469.7 L81.6,469.6 L82.0,469.6 L82.4,469.5 L82.8,469.5 L83.1,469.5
 L83.5,469.4 L83.9,469.4 L84.3,469.3 L84.7,469.3 L85.1,469.2 L85.5,469.2 L85.9,469.1 L86.2,469.1
 L86.6,469.0 L87.0,469.0 L87.4,468.9 L87.8,468.9 L88.2,468.8 L88.6,468.7 L89.0,468.7 L89.3,468.6
 L89.7,468.5 L90.1,468.5 L90.5,468.4 L90.9,468.3 L91.3,468.3 L91.7,468.2 L92.1,468.1 L92.4,468.1
 L92.8,468.0 L93.2,467.9 L93.6,467.8 L94.0,467.7 L94.4,467.7 L94.8,467.6 L95.2,467.5 L95.5,467.4
 L95.9,467.3 L96.3,467.2 L96.7,467.1 L97.1,467.1 L97.5,467.0 L97.9,466.9 L98.3,466.8 L98.6,466.7
 L99.0,466.6 L99.4,466.5 L99.8,466.4 L100.2,466.3 L100.6,466.2 L101.0,466.1 L101.4,466.0 L101.7,465.9
 L102.1,465.8 L102.5,465.6 L102.9,465.5 L103.3,465.4 L103.7,465.3 L104.1,465.2 L104.5,465.1 L104.8,465.0
 L105.2,464.9 L105.6,464.7 L106.0,464.6 L106.4,464.5 L106.8,464.4 L107.2,464.2 L107.6,464.1 L107.9,464.0
 L108.3,463.9 L108.7,463.7 L109.1,463.6 L109.5,463.5 L109.9,463.3 L110.3,463.2 L110.7,463.1 L111.0,463.1
 L111.4,462.9 L111.8,462.8 L112.2,462.7 L112.6,462.5 L113.0,462.4 L113.4,462.2 L113.8,462.1 L114.1,461.9
 L114.5,461.8 L114.9,461.6 L115.3,461.5 L115.7,461.3 L116.1,461.2 L116.5,461.0 L116.9,460.9 L117.2,460.7
 L117.6,460.6 L118.0,460.4 L118.4,460.3 L118.8,460.1 L119.2,459.9 L119.6,459.8 L120.0,459.6 L120.3,459.5
 L120.7,459.3 L121.1,459.1 L121.5,459.0 L121.9,458.8 L122.3,458.6 L122.7,458.4 L123.1,458.3 L123.4,458.1
 L123.8,457.9 L124.2,457.7 L124.6,457.6 L125.0,457.4 L125.4,457.2 L125.8,457.0 L126.2,456.9 L126.5,456.7
 L126.9,456.5 L127.3,456.3 L127.7,456.1 L128.1,455.9 L128.5,455.7 L128.9,455.6 L129.3,455.4 L129.6,455.2
 L130.0,455.0 L130.4,454.8 L130.8,454.6 L131.2,454.4 L131.6,454.2 L132.0,454.0 L132.4,453.8 L132.7,453.6
 L133.1,453.4 L133.5,453.2 L133.9,453.0 L134.3,452.8 L134.7,452.6 L135.1,452.4 L135.5,452.2 L135.8,452.0
 L136.2,451.8 L136.6,451.6 L137.0,451.3 L137.4,451.1 L137.8,450.9 L138.2,450.7 L138.6,450.5 L138.9,450.3
 L139.3,450.1 L139.7,449.8 L140.1,449.6 L140.5,449.4 L140.9,449.2 L141.3,449.0 L141.7,448.7 L142.0,448.5
 L142.4,448.3 L142.8,448.1 L143.2,447.8 L143.6,447.6 L144.0,447.4 L144.4,447.1 L144.8,446.9 L145.1,446.7
 L145.5,446.5 L145.9,446.2 L146.3,446.0 L146.7,445.7 L147.1,445.5 L147.5,445.3 L147.9,445.0 L148.2,444.8
 L148.6,444.6 L149.0,444.3 L149.4,444.1 L149.8,444.1 L150.2,443.8 L150.6,443.6 L151.0,443.3 L151.3,443.1
 L151.7,442.9 L152.1,442.6 L152.5,442.4 L152.9,442.1 L153.3,441.9 L153.7,441.6 L154.1,441.3 L154.4,441.1
 L154.8,440.8 L155.2,440.6 L155.6,440.3 L156.0,440.1 L156.4,439.8 L156.8,439.6 L157.2,439.3 L157.5,439.0
 L157.9,438.8 L158.3,438.5 L158.7,438.3 L159.1,438.0 L159.5,437.7 L159.9,437.5 L160.3,437.2 L160.6,436.9
 L161.0,436.7 L161.4,436.4 L161.8,436.1 L162.2,435.9 L162.6,435.6 L163.0,435.3 L163.4,435.0 L163.7,434.8
 L164.1,434.5 L164.5,434.2 L164.9,433.9 L165.3,433.7 L165.7,433.4 L166.1,433.1 L166.5,432.8 L166.8,432.6
 L167.2,432.3 L167.6,432.0 L168.0,431.7 L168.4,431.4 L16

In [ ]:
//ISessionInfoExtensions.PlotData(evalDataRef.ElementAt(5), "time", "rise velocity")
GetBenchmarkQuantityComparisonOverTimeGnuplot(benchmarkData, 3, "rise velocity", dataRef).PlotSVG()

Using gnuplot: C:\Users\smuda\AppData\Local\FDY\BoSSS\bin\native\win\gnuplot-gp510-20160418-win32-mingw\gnuplot\bin\gnuplot.exe
Note: In a Jupyter Worksheet, you must NOT have a trailing semicolon in order to see the plot on screen; otherwise, the output migth be surpressed.!


<?xml version="1.0" encoding="utf-8" standalone="no"?>
<!DOCTYPE svg PUBLIC "-//W3C//DTD SVG 1.1//EN"
 "http://www.w3.org/Graphics/SVG/1.1/DTD/svg11.dtd">
 

 Gnuplot 
 Produced by GNUPLOT 5.1 patchlevel 0 

 

 
 

 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 -0.05 
 
 
 
 
 0 
 
 
 
 
 0.05 
 
 
 
 
 0.1 
 
 
 
 
 0.15 
 
 
 
 
 0.2 
 
 
 
 
 0.25 
 
 
 
 
 0 
 
 
 
 
 0.5 
 
 
 
 
 1 
 
 
 
 
 1.5 
 
 
 
 
 2 
 
 
 
 
 2.5 
 
 
 
 
 3 
 
 
 
 
 3.5 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 
 rise velocity 
 
 
 
 
 time 
 
 
 
 
 BoSSS 
 
 
 
 
 BoSSS 
 
 
 
	<path stroke='rgb( 0, 0, 255)' d='M555.2,57.1 L608.6,57.1 M80.2,458.0 L80.6,456.2 L81.0,454.4 L81.3,452.6 L81.7,450.8 L82.1,449.1
 L82.5,447.3 L82.9,445.5 L83.3,443.8 L83.6,442.0 L84.0,440.3 L84.4,438.5 L84.8,436.8 L85.2,435.0
 L85.6,433.3 L85.9,431.6 L86.3,429.9 L86.7,428.2 L87.1,426.4 L87.5,424.7 L87.9,423.0 L88.2,421.3
 L88.6,419.7 L89.0,418.0 L89.4,416.3 L89.8,414.6 L90.2,412.9 L90.5,411.3 L90.9,409.6 L91.3,407.9
 L91.7,406.3 L92.1,404.6 L92.4,403.0 L92.8,401.3 L93.2,399.7 L93.6,398.1 L94.0,396.4 L94.4,394.8
 L94.7,393.2 L95.1,391.5 L95.5,389.9 L95.9,388.3 L96.3,386.7 L96.7,385.1 L97.0,383.5 L97.4,381.9
 L97.8,380.3 L98.2,378.7 L98.6,377.1 L99.0,375.5 L99.3,373.9 L99.7,372.4 L100.1,370.8 L100.5,369.2
 L100.9,367.6 L101.3,366.1 L101.6,364.5 L102.0,363.0 L102.4,361.4 L102.8,359.9 L103.2,358.3 L103.6,356.8
 L103.9,355.2 L104.3,353.7 L104.7,352.2 L105.1,350.6 L105.5,349.1 L105.8,347.6 L106.2,346.0 L106.6,344.5
 L107.0,343.0 L107.4,341.5 L107.8,340.0 L108.1,338.5 L108.5,337.0 L108.9,335.5 L109.3,334.0 L109.7,332.5
 L110.1,331.0 L110.4,329.5 L110.8,328.0 L111.2,326.6 L111.6,325.1 L112.0,323.6 L112.4,322.2 L112.7,320.7
 L113.1,319.2 L113.5,317.8 L113.9,316.3 L114.3,314.9 L114.7,313.4 L115.0,312.0 L115.4,310.5 L115.8,309.1
 L116.2,307.6 L116.6,306.2 L116.9,304.8 L117.3,303.3 L117.7,301.9 L118.1,300.5 L118.5,299.1 L118.9,297.7
 L119.2,296.2 L119.6,294.8 L120.0,293.4 L120.4,292.0 L120.8,290.6 L121.2,289.2 L121.5,287.8 L121.9,286.4
 L122.3,285.1 L122.7,283.7 L123.1,282.3 L123.5,280.9 L123.8,279.5 L124.2,278.2 L124.6,276.8 L125.0,275.4
 L125.4,274.1 L125.8,272.7 L126.1,271.4 L126.5,270.0 L126.9,268.7 L127.3,267.3 L127.7,266.0 L128.1,264.6
 L128.4,263.3 L128.8,262.0 L129.2,260.6 L129.6,259.3 L130.0,258.0 L130.3,256.7 L130.7,255.4 L131.1,254.1
 L131.5,252.8 L131.9,251.4 L132.3,250.1 L132.6,248.8 L133.0,247.6 L133.4,246.3 L133.8,245.0 L134.2,243.7
 L134.6,242.4 L134.9,241.1 L135.3,239.9 L135.7,238.6 L136.1,237.3 L136.5,236.1 L136.9,234.8 L137.2,233.5
 L137.6,232.3 L138.0,231.0 L138.4,229.8 L138.8,228.5 L139.2,227.3 L139.5,226.1 L139.9,224.8 L140.3,223.6
 L140.7,222.4 L141.1,221.2 L141.4,220.0 L141.8,218.7 L142.2,217.5 L142.6,216.3 L143.0,215.1 L143.4,213.9
 L143.7,212.7 L144.1,211.5 L144.5,210.4 L144.9,209.2 L145.3,208.0 L145.7,206.8 L146.0,205.6 L146.4,204.5
 L146.8,203.3 L147.2,202.1 L147.6,201.0 L148.0,199.8 L148.3,198.7 L148.7,197.5 L149.1,196.4 L149.5,195.3
 L149.9,194.1 L150.3,193.0 L150.6,191.9 L151.0,190.8 L151.4,189.6 L151.8,188.5 L152.2,187.4 L152.5,186.3
 L152.9,185.2 L153.3,184.1 L153.7,183.0 L154.1,181.9 L154.5,180.9 L154.8,179.8 L155.2,178.7 L155.6,177.6
 L156.0,176.6 L156.4,175.5 L156.8,174.4 L157.1,173.3 L157.5,172.2 L157.9,171.2 L158.3,170.1 L158.7,169.1
 L159.1,168.1 L159.4,167.0 L159.8,166.0 L160.2,165.0 L160.6,164.0 L161.0,163.0 L161.4,162.0 L161.7,160.9
 L162.1,160.0 L162.5,159.0 L162.9,158.0 L163.3,157.0 L163.7,156.0 L164.0,155.0 L164.4,154.0 L164.8,153.1
 L165.2,152.1 L165.6,151.2 L165.9,150.2 L166.3,149.3 L166.7,148.3 L167.1,147.4 L167.5,146.4 L167.9,145.5
 L168.2,144.6 L168.6,143.6 L169.0,142.7 L169.4,141.8 L169.8,140.9 L170.2,140.0 L170.5,139.1 L170.9,138.2
 L171.3,137.3 L171.7,136.4 L172.1,135.6 L172.5,134.7 L172.8,133.8 L173.2,133.0 L173.6,132.1 L174.0,131.2
 L174.4,130.4 L174.8,129.5 L175.1,128.7 L175.5,127.9 L175.9

## Error norms

In [ ]:
public static double GetRefValueInTime(Plot2Ddata.XYvalues refData, double time) {

    double[] abs = refData.Abscissas;
    double[] val = refData.Values;

    double refVal = 0.0;
    double time_i = abs[0];
    for (int i = 1; i < abs.Length; i++) {
        // if (abs[i] == time) {
        //     return val[i];
        // }
        if (abs[i] >= time) {
            // compute ref value
            double dt = abs[i] - time_i;
            double dx = val[i] - val[i-1];
            refVal = val[i-1] + time * (dx / dt);
            break;
        } else {
            time_i = abs[i];
        }

    }

    return refVal;
}

In [ ]:
// l1-error norm
public static (double l1_errNorm, double l2_errNorm, double lInf_errNorm) ComputeErrorNorms(Plot2Ddata.XYvalues data, Plot2Ddata.XYvalues refData) {

    double q_i = 0.0;
    double qRef_i = 0.0;
    double[] absc = data.Abscissas;
    double[] dqS = new double[absc.Length];
    double[]qRefS = new double[absc.Length];
    for (int i = 0; i < absc.Length; i++) {
        q_i = data.Values[i];
        qRef_i = GetRefValueInTime(refData, absc[i]);
        dqS[i] = Math.Abs(q_i - qRef_i);
        qRefS[i] = Math.Abs(qRef_i);
    }

    double l1err = dqS.Sum() / qRefS.Sum();
    double l2err = dqS.L2Norm() / qRefS.Sum();  
    double lInferr = dqS.Max() / qRefS.Max();

    return (l1err, l2err, lInferr);
}

In [ ]:
ComputeErrorNorms(benchmarkData.ElementAt(3).dataGroups[0], dataRef[3, 0].Last().dataGroups[0])

Item1,Item2,Item3
0.26746843827142003,0.008360921614515897,0.39499740487807844


## Check values at specifc points in time

## Check terminal interface shape